# Notebook 2: Experiments
2 embeddings × 2 LLMs × 3 prompts = 12 combinations

In [2]:
import json
import time
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from transformers import pipeline
from tqdm import tqdm

/Users/arystankamalov/Desktop/Minecraft_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
client = PersistentClient(path="./db")

emb_minilm = SentenceTransformer("all-MiniLM-L6-v2")
emb_mpnet  = SentenceTransformer("all-mpnet-base-v2")

col_minilm = client.get_collection("minecraft_minilm")
col_mpnet  = client.get_collection("minecraft_mpnet")

llm_qwen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=-1)
llm_tiny = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct", device=-1)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9385.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6745.82it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5361.95it/s]


In [4]:
prompt_minimal = """Context: {context}
Question: {question}
Answer:"""

prompt_detailed = """You are a Minecraft expert assistant.
Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
Answer:"""

prompt_cot = """You are a Minecraft expert.
Think step by step using the context provided.

Context:
{context}

Question: {question}
Let's think step by step:"""

test_questions = [
    "How do I craft a sword in Minecraft?",
    "What does a creeper drop when killed?",
    "How do I build a nether portal?",
    "What is the best way to find diamonds?",
    "How do I tame a wolf in Minecraft?",
]

In [5]:
def retrieve(collection, emb_model, question, top_k=3):
    query_embedding = emb_model.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    context = "\n".join([line for line in results['documents'][0]])
    return context

def generate(llm, prompt):
    out = llm(prompt, max_new_tokens=150, do_sample=False, repetition_penalty=1.1, truncation=True)
    return out[0]["generated_text"][len(prompt):].strip()

In [6]:
embedding_options = [
    ("minilm", emb_minilm, col_minilm),
    ("mpnet",  emb_mpnet,  col_mpnet),
]

llm_options = [
    ("qwen",    llm_qwen),
    ("smollm",  llm_tiny),
]

prompt_options = [
    ("minimal",  prompt_minimal),
    ("detailed", prompt_detailed),
    ("cot",      prompt_cot),
]

all_results = []

for emb_name, emb_model, col in embedding_options:
    for llm_name, llm in llm_options:
        for prompt_name, prompt_template in prompt_options:

            exp_name = f"{emb_name} + {llm_name} + {prompt_name}"
            print("Running:", exp_name)

            latencies = []
            samples = []

            for question in test_questions:
                t0 = time.time()
                context = retrieve(col, emb_model, question)
                prompt = prompt_template.format(context=context, question=question)
                answer = generate(llm, prompt)
                latencies.append(time.time() - t0)
                samples.append({"question": question, "answer": answer})

            avg_latency = sum(latencies) / len(latencies)
            print(f"  avg latency: {avg_latency:.2f}s")

            all_results.append({
                "exp_name":    exp_name,
                "embedding":   emb_name,
                "llm":         llm_name,
                "prompt":      prompt_name,
                "top_k":       3,
                "avg_latency": avg_latency,
                "samples":     samples,
            })

print("Total experiments:", len(all_results))

Running: minilm + qwen + minimal


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have be

  avg latency: 5.20s
Running: minilm + qwen + detailed


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  avg latency: 3.75s
Running: minilm + qwen + cot


...

  avg latency: 1.37s
Running: mpnet + smollm + detailed


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  avg latency: 1.43s
Running: mpnet + smollm + cot


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  avg latency: 1.34s
Total experiments: 12


In [7]:
# Top-K sweep on best (fastest) combo
best = min(all_results, key=lambda x: x["avg_latency"])
print("Best combo:", best["exp_name"])

best_emb_name, best_emb_model, best_col = next(e for e in embedding_options if e[0] == best["embedding"])
best_llm_name, best_llm = next(l for l in llm_options if l[0] == best["llm"])
best_prompt_name, best_prompt_template = next(p for p in prompt_options if p[0] == best["prompt"])

for top_k in [1, 3, 5]:
    latencies = []
    for question in test_questions:
        t0 = time.time()
        context = retrieve(best_col, best_emb_model, question, top_k)
        prompt = best_prompt_template.format(context=context, question=question)
        answer = generate(best_llm, prompt)
        latencies.append(time.time() - t0)
    avg = sum(latencies) / len(latencies)
    print(f"top_k={top_k}  avg latency={avg:.2f}s")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Best combo: minilm + smollm + cot


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


top_k=1  avg latency=1.16s


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

top_k=3  avg latency=1.53s


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


top_k=5  avg latency=1.93s


In [8]:
import os
os.makedirs("results", exist_ok=True)

with open("results/experiments.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("saved to results/experiments.json")

saved to results/experiments.json
